# GVC Measure 

Pulls OECD Trade-in-Value-Added (TiVa) API from each sector mapping to HS-6 Product Level in order to develop and create a GVC-intensity scoring measure.



In [11]:
# Pull BACII data from the database for 1992


import datetime
import os
import sys
import time 
import numpy as np
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import create_engine
from sqlalchemy import MetaData, Table, Column, Integer, String, DateTime, Float
from sqlalchemy.sql import select, and_, or_, not_
from sqlalchemy import func
from sqlalchemy import text
from sqlalchemy import inspect
from sqlalchemy import exc  
from sqlalchemy import event
from sqlalchemy.pool import Pool
import logging

logging.basicConfig()
logging.getLogger('sqlalchemy.engine').setLevel(logging.INFO)

## Data Loader for Project Codes

In [12]:
import sys
from pathlib import Path

# Works on Mac, Windows, Linux — no hardcoded paths
PROJECT_ROOT = Path.cwd().parent  # adjust depth if no tebook is in a subfolder
sys.path.insert(0, str(PROJECT_ROOT))

from config import DATA_DIR, OUTPUT_DIR

# Use like this — path separators handled automatically
baci_file = DATA_DIR / "BACI_HS92_V202601" / "product_codes_HS92_V202601.csv"

In [13]:
import pandas as pd

def load_product_codes():
    # Load HS92 product codes from data folder
    file_path = 'data/BACI_HS92_V202601/product_codes_HS92_V202601.csv'

    try:
        df = pd.read_csv(file_path, dtype='str')
        # Fix leading zeros
        df['code'] = df['code'].str.zfill(6)
        # Drop any non-numeric legacy rows (e.g. 9999AA)
        df = df[df['code'].str.match(r'^\d{6}$')]
        # Check all codes are exactly 6 digits
        lengths = df['code'].str.len().value_counts()
        print("Code length distribution:")
        print(lengths)
        print(f"Loaded {len(df)} product codes")
        print(df.head(20))
        return df

    except FileNotFoundError:
        print(f"File not found: {file_path}")
        import os
        data_dir = 'data/BACI_HS92_V202601'
        if os.path.exists(data_dir):
            print(os.listdir(data_dir))

df_hs92 = load_product_codes()


Code length distribution:
code
6    5021
Name: count, dtype: int64
Loaded 5021 product codes
      code                                        description
0   010111           Horses: live, pure-bred breeding animals
1   010119  Horses: live, other than pure-bred breeding an...
2   010120                     Asses, mules and hinnies: live
3   010210   Bovine animals: live, pure-bred breeding animals
4   010290  Bovine animals: live, other than pure-bred bre...
5   010310            Swine: live, pure-bred breeding animals
6   010391  Swine: live, (other than pure-bred breeding an...
7   010392  Swine: live, (other than pure-bred breeding an...
8   010410                                        Sheep: live
9   010420                                        Goats: live
10  010511  Poultry: live, fowls of the species gallus dom...
11  010519  Poultry: live, weighing not more than 185g, du...
12  010591  Poultry: live, fowls of the species gallus dom...
13  010599  Poultry: live, ducks, geese

## HS Chapter, Subheadings, Descriptions

In [14]:
import requests
import pandas as pd

# ── Layer 0 Chapter Prior ─────────────────────────────────────────────────────
CLASS_NAMES = {
    1: "Raw/Primary",
    2: "Processed Material",
    3: "Component/Part",
    4: "Intermediate Assembly/Capital",
    5: "Final Good",
}

CHAPTER_PRIOR = [
    ( 1, 27, 1),   # live animals, food, raw materials, minerals, fuels
    (28, 40, 2),   # chemicals, plastics, rubber
    (41, 60, 3),   # leather, wood, paper, textiles (fabrics/yarns)
    (61, 63, 5),   # apparel and clothing accessories
    (64, 67, 5),   # footwear, headgear
    (68, 83, 3),   # stone, glass, ceramics, base metals, metal articles
    (84, 92, 4),   # machinery, electrical equipment, instruments
    (93, 93, 4),   # arms and ammunition
    (94, 96, 5),   # furniture, toys, miscellaneous manufactures
    (97, 97, 5),   # art, antiques
]

chapter_prior = pd.DataFrame([
    {"chapter": str(ch).zfill(2), "prior_class": cls, "prior_label": CLASS_NAMES[cls]}
    for lo, hi, cls in CHAPTER_PRIOR
    for ch in range(lo, hi + 1)
])

print("Chapter prior class distribution:")
print(chapter_prior.groupby(["prior_class", "prior_label"])["chapter"]
      .count().rename("n_chapters").to_string())

# ── Source 1: GitHub CSV → chapter and heading text ───────────────────────────
print("\nFetching GitHub HS hierarchy CSV...")
gh = pd.read_csv(
    "https://raw.githubusercontent.com/datasets/harmonized-system/main/data/harmonized-system.csv",
    dtype=str
)
gh["level"] = gh["level"].str.strip()

chapter_text = (gh[gh["level"] == "2"]
    .rename(columns={"hscode": "chapter", "description": "chapter_text"})
    [["chapter", "chapter_text"]])

heading_text = (gh[gh["level"] == "4"]
    .rename(columns={"hscode": "heading", "description": "heading_text"})
    [["heading", "heading_text"]])

print(f"  Chapters: {len(chapter_text)}, Headings: {len(heading_text)}")

# ── Source 2: HS92 directly (no WITS) ────────────────────────────────────────
# Use df_hs92 loaded above — clean single-revision codes, no multi-year noise.
df_sub = (df_hs92
    .rename(columns={"code": "subheading", "description": "subheading_text"})
    .copy())
df_sub["chapter"] = df_sub["subheading"].str[:2]
df_sub["heading"]  = df_sub["subheading"].str[:4]
print(f"  Subheadings (HS92): {len(df_sub):,}")   # should be 5,022

# ── HS92 heading coverage check + OEC patch ───────────────────────────────────
hs92_headings   = df_hs92["code"].str[:4].unique()
github_headings = set(heading_text["heading"].values)
missing_hd      = [h for h in hs92_headings if h not in github_headings]
print(f"\nHS92 headings not in GitHub CSV: {len(missing_hd)}")
print(sorted(missing_hd))

if missing_hd:
    # OEC API returns {"members": [{"key": 10101, "caption": "Horses"}, ...]}
    oec_resp = requests.get(
        "https://api-v2.oec.world/tesseract/members?cube=trade_i_baci_a_92&level=HS4",
        timeout=30
    ).json()["members"]
    oec = pd.DataFrame(oec_resp)
    oec["heading"] = oec["key"].astype(str).str[-4:].str.zfill(4)
    oec = oec.rename(columns={"caption": "heading_text"})
    patch = oec[oec["heading"].isin(missing_hd)][["heading", "heading_text"]]
    heading_text = pd.concat([heading_text, patch], ignore_index=True).drop_duplicates("heading")
    print(f"Patched {len(patch)} HS92 headings from OEC")

# ── Assemble Layer 0 ──────────────────────────────────────────────────────────
df_layer0 = (df_sub
    .merge(chapter_text,  on="chapter", how="left")
    .merge(heading_text,   on="heading",  how="left")
    .merge(chapter_prior,  on="chapter",  how="left")
    [["chapter", "chapter_text",
      "heading",  "heading_text",
      "subheading", "subheading_text",
      "prior_class", "prior_label"]])

# ── Sanity checks ─────────────────────────────────────────────────────────────
print(f"\nRows:                  {len(df_layer0):,}")
print(f"Missing chapter text:  {df_layer0['chapter_text'].isna().sum()}")
print(f"Missing heading text:  {df_layer0['heading_text'].isna().sum()}")
print(f"Unassigned prior:      {df_layer0['prior_class'].isna().sum()}")
print(f"\nPrior class distribution (products):")
print(df_layer0.groupby(["prior_class","prior_label"], dropna=False)["subheading"]
      .count().rename("n_products").to_string())

df_layer0.head(10)


Chapter prior class distribution:
prior_class  prior_label                  
1            Raw/Primary                      27
2            Processed Material               13
3            Component/Part                   36
4            Intermediate Assembly/Capital    10
5            Final Good                       11

Fetching GitHub HS hierarchy CSV...
  Chapters: 97, Headings: 1229
  Subheadings (HS92): 5,021

HS92 headings not in GitHub CSV: 37
['0503', '0509', '1402', '1403', '1519', '2527', '2838', '2848', '2851', '4108', '4109', '4110', '4111', '4204', '4815', '5304', '6503', '6908', '7012', '7414', '7416', '7417', '7803', '7805', '7906', '8004', '8005', '8006', '8107', '8469', '8520', '8803', '9009', '9203', '9204', '9501', '9502']
Patched 37 HS92 headings from OEC

Rows:                  5,021
Missing chapter text:  0
Missing heading text:  0
Unassigned prior:      1

Prior class distribution (products):
prior_class  prior_label                  
1.0          Raw/Primary    

,chapter,chapter_text,heading,heading_text,subheading,subheading_text,prior_class,prior_label
0,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010111,"Horses: live, pure-bred breeding animals",1.0,Raw/Primary
1,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010119,"Horses: live, other than pure-bred breeding an...",1.0,Raw/Primary
2,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010120,"Asses, mules and hinnies: live",1.0,Raw/Primary
3,01,Animals; live,0102,Bovine animals; live,010210,"Bovine animals: live, pure-bred breeding animals",1.0,Raw/Primary
4,01,Animals; live,0102,Bovine animals; live,010290,"Bovine animals: live, other than pure-bred bre...",1.0,Raw/Primary
5,01,Animals; live,0103,Swine; live,010310,"Swine: live, pure-bred breeding animals",1.0,Raw/Primary
6,01,Animals; live,0103,Swine; live,010391,"Swine: live, (other than pure-bred breeding an...",1.0,Raw/Primary
7,01,Animals; live,0103,Swine; live,010392,"Swine: live, (other than pure-bred breeding an...",1.0,Raw/Primary
8,01,Animals; live,0104,Sheep and goats; live,010410,Sheep: live,1.0,Raw/Primary
9,01,Animals; live,0104,Sheep and goats; live,010420,Goats: live,1.0,Raw/Primary


## Layer 1 — Zero-Shot NLI on HS6 Descriptions

Runs each HS6 short description through a zero-shot Natural Language Inference (NLI) model against five chain-position hypotheses (one per class). No training data is required — classification is driven entirely by the semantic relationship between the product description and the explicitly stated hypothesis.

**Output columns added to `df_layer1`:**
- `nli_prob_1` … `nli_prob_5` — normalised entailment probability per class  
- `nli_continuous` — probability-weighted average of class indices [1, 5]  
- `nli_class` — argmax predicted class (1–5)  
- `nli_label` — human-readable class name  
- `nli_uncertainty` — normalised entropy H / log(5), bounded [0, 1]

**Models:** `cross-encoder/nli-deberta-v3-small` (primary, open weights).  
`facebook/bart-large-mnli` is defined as a second model for ensemble comparison in later work.  
Both are run locally with no API dependency.

In [15]:
# ── Layer 1: Hypothesis definitions ──────────────────────────────────────────
# Each hypothesis is the natural-language statement of what it means for a
# product to belong to that chain-position class. Phrasing is directly from
# the Master Plan §4 Layer 1. Tested empirically against anchor products —
# performance is sensitive to wording, so treat these as v1 to be iterated.

HYPOTHESES = {
    1: (
        "This is a raw or primary commodity — such as a live animal, crude oil, iron ore, "
        "raw cotton, or uncut timber — extracted or harvested in its natural state with "
        "no significant industrial transformation."
    ),
    2: (
        "This is a processed industrial material in bulk or standardised form — such as "
        "refined metal, chemical compound, plastic resin, or textile yarn — transformed "
        "from a raw input but not yet shaped into a discrete part or component."
    ),
    3: (
        "This is a discrete manufactured part, component, or accessory — such as a bearing, "
        "valve, circuit board, or engine part — described using terms like 'parts', "
        "'parts and accessories', or 'for use in', and designed to be assembled into "
        "a larger product."
    ),
    4: (
        "This is a complete functional machine, engine, or industrial apparatus — such as "
        "a diesel engine, industrial robot, machine tool, or compressor — used by producers "
        "to manufacture goods or deliver services, not sold to final consumers."
    ),
    5: (
        "This is a finished consumer good ready for direct use — such as a passenger car, "
        "smartphone, washing machine, clothing, bottled wine, or packaged food — requiring "
        "no further industrial processing before use by the end user."
    ),
}

CLASS_NAMES = {
    1: "Raw/Primary",
    2: "Processed Material",
    3: "Component/Part",
    4: "Intermediate Assembly/Capital",
    5: "Final Good",
}

print("Hypotheses defined:")
for cls, hyp in HYPOTHESES.items():
    print(f"  Class {cls} ({CLASS_NAMES[cls]}): {hyp[:60]}...")

Hypotheses defined:
  Class 1 (Raw/Primary): This is a raw or primary commodity — such as a live animal, ...
  Class 2 (Processed Material): This is a processed industrial material in bulk or standardi...
  Class 3 (Component/Part): This is a discrete manufactured part, component, or accessor...
  Class 4 (Intermediate Assembly/Capital): This is a complete functional machine, engine, or industrial...
  Class 5 (Final Good): This is a finished consumer good ready for direct use — such...


In [16]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "transformers", "sentencepiece"],
    capture_output=True, text=True
)
print(result.stdout[-3000:])
print(result.stderr[-3000:])

 ./.venv/lib/python3.13/site-packages (from transformers) (2026.4.4)


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip



In [17]:
## ── Layer 1: Load NLI model ───────────────────────────────────────────────────
# Primary model: DeBERTa-v3-small fine-tuned on MNLI + SNLI + ANLI.
# Chosen for: strong zero-shot NLI accuracy, small footprint (~180MB),
# locally runnable, open weights (Apache 2.0).
# Fallback / ensemble comparison model: facebook/bart-large-mnli.
# Not loaded by default — uncomment to run ensemble (slower, ~1.6GB).
# multi_label=True: each hypothesis is scored independently rather than
# as competing candidates. This is correct here because a description can
# plausibly entail more than one hypothesis at partial probability —
# we normalise the raw scores to a probability distribution ourselves.

#  Run this in its own cell before loading any model
# device.py  (project root, committed to GitHub)
import torch
from transformers import pipeline

def get_device() -> str:
    if torch.cuda.is_available(): return "cuda"
    if torch.backends.mps.is_available(): return "mps"
    return "cpu"

DEVICE = get_device()
print(f"Device: {DEVICE}")

if "nli" not in dir() or nli is None:
    model_id = "MoritzLaurer/deberta-v3-base-zeroshot-v1"
    print(f"Loading {model_id}...")
    nli = pipeline(
        "zero-shot-classification",
        model=model_id,
        device=DEVICE,
    )
    print("Model ready.")

# ── Updated scoring function ──────────────────────────────────────────────────
def nli_score_product(description: str) -> dict:
    result = nli(
        description,
        candidate_labels=list(HYPOTHESES.values()),
        multi_label=False,    # softmax — forces single class selection
    )
    # Map scores back to class numbers
    hyp_to_class = {v: k for k, v in HYPOTHESES.items()}
    scores = {
        hyp_to_class[label]: score
        for label, score in zip(result["labels"], result["scores"])
    }
    # Continuous score: probability-weighted average
    continuous = sum(cls * p for cls, p in scores.items())
    # Uncertainty: entropy of distribution
    import math
    entropy = -sum(p * math.log(p + 1e-9) for p in scores.values())
    uncertainty = entropy / math.log(len(scores))   # normalise to [0,1]

    predicted_class = max(scores, key=scores.get)
    return {
        "nli_class":      predicted_class,
        "nli_label":      CLASS_NAMES[predicted_class],
        "nli_continuous": round(continuous, 3),
        "nli_uncertainty": round(uncertainty, 3),
        "class_probs":    {cls: round(p, 4) for cls, p in sorted(scores.items())},
    }

Device: mps


In [18]:
# ── Layer 1: Scoring function ─────────────────────────────────────────────────
import math

def nli_score_product(description: str) -> dict:
    """
    Score one HS6 product description against all five chain-position hypotheses.

    Returns a dict with:
      nli_prob_1..5   normalised entailment probability per class
      nli_continuous  probability-weighted average of class indices [1, 5]
      nli_class       predicted class (int 1-5, argmax)
      nli_label       human-readable class name
      nli_uncertainty normalised entropy H / log(5), bounded [0, 1]

    Returns all-None dict for empty or null descriptions.
    """
    desc = str(description).strip() if description and str(description).strip() else None
    if desc is None:
        return {
            "nli_prob_1": None, "nli_prob_2": None, "nli_prob_3": None,
            "nli_prob_4": None, "nli_prob_5": None,
            "nli_continuous":  None,
            "nli_class":       None,
            "nli_label":       None,
            "nli_uncertainty": None,
        }

    # multi_label=True: independent entailment score per hypothesis
    result = nli(
        desc,
        candidate_labels=list(HYPOTHESES.values()),
        multi_label=False,
    )

    # Map hypothesis text back to class number
    hyp_to_class = {v: k for k, v in HYPOTHESES.items()}
    raw = {hyp_to_class[label]: score
           for label, score in zip(result["labels"], result["scores"])}

    # Normalise raw entailment scores to sum to 1
    total = sum(raw.values())
    norm  = {cls: raw[cls] / total for cls in sorted(raw)}

    # Continuous score: probability-weighted average of class indices
    continuous = sum(cls * norm[cls] for cls in norm)

    # Predicted class: argmax of normalised probabilities
    predicted = max(norm, key=norm.get)

    # Uncertainty: normalised Shannon entropy H / log(5), bounded [0, 1]
    # High uncertainty (→ 1.0) = model spreads probability evenly across classes
    # Low uncertainty (→ 0.0)  = model concentrates on one class
    entropy     = -sum(p * math.log(p + 1e-12) for p in norm.values())
    max_entropy = math.log(len(norm))
    uncertainty = entropy / max_entropy if max_entropy > 0 else 0.0

    return {
        "nli_prob_1":      round(norm[1], 6),
        "nli_prob_2":      round(norm[2], 6),
        "nli_prob_3":      round(norm[3], 6),
        "nli_prob_4":      round(norm[4], 6),
        "nli_prob_5":      round(norm[5], 6),
        "nli_continuous":  round(continuous, 4),
        "nli_class":       predicted,
        "nli_label":       CLASS_NAMES[predicted],
        "nli_uncertainty": round(uncertainty, 4),
    }

print("nli_score_product() defined.")

nli_score_product() defined.


In [19]:
# ── Layer 1: Anchor product smoke test ───────────────────────────────────────
# Run a small set of unambiguous anchor products before scoring the full
# HS6 universe. Every anchor must be classified correctly — any failure
# here is a hard problem with the hypothesis phrasing, not a borderline case.
# Anchor list from Master Plan §6.

ANCHORS = [
    # (description, expected_class)
    ("Petroleum oils, crude",                               1),
    ("Iron ores and concentrates, non-agglomerated",        1),
    ("Cotton, not carded or combed",                        1),
    ("Copper, refined, in the form of cathodes",            2),
    ("Steel, flat-rolled, cold-rolled, width>=600mm",       2),
    ("Polyethylene having a specific gravity of <0.94",     2),
    ("Ball or roller bearings",                             3),
    ("Printed circuits",                                    3),
    ("Electric motors of an output exceeding 37.5 W",      3),
    ("Compression-ignition internal combustion engines",    4),
    ("Machine-tools for working metal by removal of material", 4),
    ("Motor cars and other motor vehicles for persons",     5),
    ("Telephone sets for cellular networks",                5),
    ("Wine of fresh grapes, in containers <=2 litres",      5),
]

print(f"Running anchor smoke test on {len(ANCHORS)} products...")
print(f"{'Description':<55} {'Exp':>4} {'Got':>4} {'Score':>6} {'Unc':>5}  OK?")
print("-" * 85)

n_correct = 0
for desc, expected in ANCHORS:
    result = nli_score_product(desc)
    got    = result["nli_class"]
    score  = result["nli_continuous"]
    unc    = result["nli_uncertainty"]
    ok     = "✓" if got == expected else "✗"
    if got == expected:
        n_correct += 1
    print(f"{desc[:54]:<55} {expected:>4} {got:>4} {score:>6.3f} {unc:>5.3f}  {ok}")

print(f"Anchor accuracy: {n_correct}/{len(ANCHORS)} = {n_correct/len(ANCHORS):.0%}")
print("Proceed to full scoring only if accuracy is high (>=80% recommended).")

Running anchor smoke test on 14 products...
Description                                              Exp  Got  Score   Unc  OK?
-------------------------------------------------------------------------------------
Petroleum oils, crude                                      1    1  2.910 0.975  ✓
Iron ores and concentrates, non-agglomerated               1    1  2.834 0.977  ✓
Cotton, not carded or combed                               1    1  2.853 0.965  ✓
Copper, refined, in the form of cathodes                   2    2  3.308 0.961  ✓
Steel, flat-rolled, cold-rolled, width>=600mm              2    2  3.266 0.950  ✓
Polyethylene having a specific gravity of <0.94            2    2  3.000 0.972  ✓
Ball or roller bearings                                    3    5  3.363 0.968  ✗
Printed circuits                                           3    5  3.249 0.968  ✗
Electric motors of an output exceeding 37.5 W              3    4  3.338 0.973  ✗
Compression-ignition internal combustion engines

In [20]:
# ── Layer 1: Score full HS92 corpus (batched) ─────────────────────────────────
# Scores all 5,022 products using the pipeline's built-in batching.
# Batching sends multiple products through the model simultaneously, which is
# ~10-20x faster than one-at-a-time on MPS/GPU.
#
# Runtime estimate (MPS, batch_size=32): ~5–10 min
# Runtime estimate (CPU,  batch_size=16): ~20–30 min

import pandas as pd
import math
from tqdm.auto import tqdm

BATCH_SIZE = 32   # reduce to 16 if you hit memory errors

texts      = df_layer0["subheading_text"].tolist()
hyp_list   = list(HYPOTHESES.values())
hyp_to_cls = {v: k for k, v in HYPOTHESES.items()}

print(f"Scoring {len(texts):,} products (batch_size={BATCH_SIZE}) ...")

# pipeline() accepts a list of strings + batch_size for automatic batching
raw_results = []
for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="NLI batches"):
    batch = texts[i : i + BATCH_SIZE]
    raw_results.extend(
        nli(batch, candidate_labels=hyp_list, multi_label=False, batch_size=BATCH_SIZE)
    )

# Post-process each result into the standard output dict
def parse_result(r):
    scores = {hyp_to_cls[label]: score
              for label, score in zip(r["labels"], r["scores"])}
    continuous      = sum(cls * p for cls, p in scores.items())
    entropy         = -sum(p * math.log(p + 1e-9) for p in scores.values())
    uncertainty     = entropy / math.log(len(scores))
    predicted_class = max(scores, key=scores.get)
    return {
        "nli_class":       predicted_class,
        "nli_label":       CLASS_NAMES[predicted_class],
        "nli_continuous":  round(continuous, 3),
        "nli_uncertainty": round(uncertainty, 3),
        **{f"nli_prob_{cls}": round(p, 6) for cls, p in sorted(scores.items())},
    }

df_nli    = pd.DataFrame([parse_result(r) for r in raw_results])
df_layer1 = pd.concat([df_layer0.reset_index(drop=True),
                        df_nli.reset_index(drop=True)], axis=1)

print(f"Layer 1 shape: {df_layer1.shape}")
print(f"Missing NLI scores: {df_layer1['nli_class'].isna().sum()}")
print(f"Predicted class distribution:")
print(
    df_layer1
    .groupby(["nli_class", "nli_label"], dropna=False)["subheading"]
    .count()
    .rename("n_products")
    .to_string()
)

out_path = "layer1_nli_scores_hs92.csv"
df_layer1.to_csv(out_path, index=False)
print(f"Saved to {out_path}")


Scoring 5,021 products (batch_size=32) ...


NLI batches: 100%|██████████| 157/157 [14:52<00:00,  5.69s/it]


Layer 1 shape: (5021, 17)
Missing NLI scores: 0
Predicted class distribution:
nli_class  nli_label                    
1          Raw/Primary                        96
2          Processed Material               1948
3          Component/Part                     25
4          Intermediate Assembly/Capital     171
5          Final Good                       2781
Saved to layer1_nli_scores_hs92.csv


In [21]:
# ── Layer 1: Score full HS92 corpus (batched) ─────────────────────────────────
# Scores all 5,022 products using the pipeline's built-in batching.
# Batching sends multiple products through the model simultaneously, which is
# ~10-20x faster than one-at-a-time on MPS/GPU.
#
# Runtime estimate (MPS, batch_size=32): ~5–10 min
# Runtime estimate (CPU,  batch_size=16): ~20–30 min

import pandas as pd
import math
from tqdm.auto import tqdm

BATCH_SIZE = 32   # reduce to 16 if you hit memory errors

texts      = df_layer0["subheading_text"].tolist()
hyp_list   = list(HYPOTHESES.values())
hyp_to_cls = {v: k for k, v in HYPOTHESES.items()}

print(f"Scoring {len(texts):,} products (batch_size={BATCH_SIZE}) ...")

# pipeline() accepts a list of strings + batch_size for automatic batching
raw_results = []
for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="NLI batches"):
    batch = texts[i : i + BATCH_SIZE]
    raw_results.extend(
        nli(batch, candidate_labels=hyp_list, multi_label=False, batch_size=BATCH_SIZE)
    )

# Post-process each result into the standard output dict
def parse_result(r):
    scores = {hyp_to_cls[label]: score
              for label, score in zip(r["labels"], r["scores"])}
    continuous      = sum(cls * p for cls, p in scores.items())
    entropy         = -sum(p * math.log(p + 1e-9) for p in scores.values())
    uncertainty     = entropy / math.log(len(scores))
    predicted_class = max(scores, key=scores.get)
    return {
        "nli_class":       predicted_class,
        "nli_label":       CLASS_NAMES[predicted_class],
        "nli_continuous":  round(continuous, 3),
        "nli_uncertainty": round(uncertainty, 3),
        **{f"nli_prob_{cls}": round(p, 6) for cls, p in sorted(scores.items())},
    }

df_nli    = pd.DataFrame([parse_result(r) for r in raw_results])
df_layer1 = pd.concat([df_layer0.reset_index(drop=True),
                        df_nli.reset_index(drop=True)], axis=1)

print(f"Layer 1 shape: {df_layer1.shape}")
print(f"Missing NLI scores: {df_layer1['nli_class'].isna().sum()}")
print(f"Predicted class distribution:")
print(
    df_layer1
    .groupby(["nli_class", "nli_label"], dropna=False)["subheading"]
    .count()
    .rename("n_products")
    .to_string()
)

out_path = "layer1_nli_scores_hs92.csv"
df_layer1.to_csv(out_path, index=False)
print(f"Saved to {out_path}")


Scoring 5,021 products (batch_size=32) ...


NLI batches: 100%|██████████| 157/157 [14:09<00:00,  5.41s/it]


Layer 1 shape: (5021, 17)
Missing NLI scores: 0
Predicted class distribution:
nli_class  nli_label                    
1          Raw/Primary                        96
2          Processed Material               1948
3          Component/Part                     25
4          Intermediate Assembly/Capital     171
5          Final Good                       2781
Saved to layer1_nli_scores_hs92.csv


In [22]:
# ── Layer 1: Diagnostics ──────────────────────────────────────────────────────
# Examines agreement between Layer 0 chapter prior and Layer 1 NLI predictions,
# and flags high-uncertainty products for inspection.

import pandas as pd

# 1. Agreement rate between chapter prior and NLI prediction
df_layer1["prior_nli_agree"] = (
    df_layer1["prior_class"] == df_layer1["nli_class"]
)
agree_rate = df_layer1["prior_nli_agree"].mean()
print(f"Chapter prior / NLI agreement rate: {agree_rate:.1%}")
print("(Expected to be moderate — NLI adds within-chapter resolution)")

# 2. Disagreement breakdown by chapter
disagree = df_layer1[~df_layer1["prior_nli_agree"]].copy()
print(f"Disagreements: {len(disagree):,} products ({len(disagree)/len(df_layer1):.1%})")
print("Top 10 chapters by disagreement count:")
print(
    disagree.groupby("chapter")["subheading"]
    .count()
    .sort_values(ascending=False)
    .head(10)
    .to_string()
)

# 3. High-uncertainty products (uncertainty > 0.75 = model very unsure)
HIGH_UNC_THRESHOLD = 0.75
high_unc = df_layer1[df_layer1["nli_uncertainty"] > HIGH_UNC_THRESHOLD].copy()
print(f"High-uncertainty products (unc > {HIGH_UNC_THRESHOLD}): {len(high_unc):,}")
print("Sample of high-uncertainty products (likely double-dipping candidates):")
print(
    high_unc[["subheading", "subheading_text", "nli_class",
               "nli_label", "nli_uncertainty"]]
    .sort_values("nli_uncertainty", ascending=False)
    .head(15)
    .to_string(index=False)
)

# 4. Continuous score distribution by predicted class
print("Continuous score statistics by NLI class:")
print(
    df_layer1
    .groupby("nli_class")["nli_continuous"]
    .agg(["mean", "std", "min", "max"])
    .round(3)
    .to_string()
)

df_layer1.head(10)

Chapter prior / NLI agreement rate: 22.6%
(Expected to be moderate — NLI adds within-chapter resolution)
Disagreements: 3,887 products (77.4%)
Top 10 chapters by disagreement count:
chapter
84    418
85    254
72    194
90    151
52    127
73    118
55    115
48    111
29    109
03     87
High-uncertainty products (unc > 0.75): 5,021
Sample of high-uncertainty products (likely double-dipping candidates):
subheading                                                                                                                                                                               subheading_text  nli_class                     nli_label  nli_uncertainty
    510320                                             Wool and hair: waste of wool or of fine animal hair, including yarn waste, but excluding garnetted stock and noils of wool or of fine animal hair          5                    Final Good            0.999
    845819                                                                

,chapter,chapter_text,heading,heading_text,subheading,subheading_text,prior_class,prior_label,nli_class,nli_label,nli_continuous,nli_uncertainty,nli_prob_1,nli_prob_2,nli_prob_3,nli_prob_4,nli_prob_5,prior_nli_agree
0,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010111,"Horses: live, pure-bred breeding animals",1.0,Raw/Primary,5,Final Good,3.216,0.987,0.193916,0.149263,0.179694,0.201248,0.275880,False
1,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010119,"Horses: live, other than pure-bred breeding an...",1.0,Raw/Primary,5,Final Good,3.089,0.997,0.181346,0.212429,0.172367,0.203494,0.230364,False
2,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010120,"Asses, mules and hinnies: live",1.0,Raw/Primary,5,Final Good,3.043,0.976,0.236675,0.202044,0.106054,0.192227,0.263000,False
3,01,Animals; live,0102,Bovine animals; live,010210,"Bovine animals: live, pure-bred breeding animals",1.0,Raw/Primary,5,Final Good,3.127,0.990,0.210952,0.168185,0.169174,0.185801,0.265888,False
4,01,Animals; live,0102,Bovine animals; live,010290,"Bovine animals: live, other than pure-bred bre...",1.0,Raw/Primary,5,Final Good,3.082,0.995,0.204986,0.189951,0.169939,0.188105,0.247019,False
5,01,Animals; live,0103,Swine; live,010310,"Swine: live, pure-bred breeding animals",1.0,Raw/Primary,5,Final Good,3.094,0.988,0.224045,0.167475,0.165524,0.175855,0.267102,False
6,01,Animals; live,0103,Swine; live,010391,"Swine: live, (other than pure-bred breeding an...",1.0,Raw/Primary,5,Final Good,3.075,0.988,0.225299,0.172069,0.170064,0.167101,0.265467,False
7,01,Animals; live,0103,Swine; live,010392,"Swine: live, (other than pure-bred breeding an...",1.0,Raw/Primary,5,Final Good,3.081,0.991,0.219670,0.174113,0.171749,0.174113,0.260356,False
8,01,Animals; live,0104,Sheep and goats; live,010410,Sheep: live,1.0,Raw/Primary,1,Raw/Primary,2.874,0.974,0.295820,0.169543,0.124041,0.186207,0.224389,True
9,01,Animals; live,0104,Sheep and goats; live,010420,Goats: live,1.0,Raw/Primary,1,Raw/Primary,2.838,0.953,0.335585,0.139756,0.118609,0.163391,0.242659,True


In [23]:
# ── Layer 1: Anchor validation — formal pass / fail ──────────────────────────
# Looks up HS92-verified anchor codes in df_layer1 after full scoring.

ANCHOR_CODES = {
    # Class 1
    "270900": 1,  # Petroleum oils, crude
    "260111": 1,  # Iron ores, non-agglomerated
    "520100": 1,  # Cotton, not carded or combed
    "270112": 1,  # Bituminous coal
    "260600": 1,  # Bauxite
    # Class 2
    "740811": 2,  # Refined copper wire
    "720917": 2,  # Cold-rolled steel >=600mm
    "390110": 2,  # Polyethylene <0.94 density
    "520511": 2,  # Cotton yarn, single, uncombed
    "760110": 2,  # Aluminium, not alloyed, unwrought
    # Class 3
    "848210": 3,  # Ball bearings
    "853400": 3,  # Printed circuits
    "850131": 3,  # DC motors >37.5W <=750W
    "848140": 3,  # Safety/relief valves
    "900190": 3,  # Lenses, optical elements
    # Class 4
    "840820": 4,  # Compression-ignition engines
    "845710": 4,  # Machining centres for metal
    "847950": 4,  # Industrial robots
    "848310": 4,  # Transmission shafts and cranks
    "841430": 4,  # Compressors for refrigerating equipment
    # Class 5
    "870321": 5,  # Motor cars <=1000cc
    "851120": 5,  # Telephone apparatus (HS92 code for phones)
    "845011": 5,  # Washing machines <=6kg
    "220421": 5,  # Wine in containers <=2 litres
    "610311": 5,  # Men's suits, of wool
    "841821": 5,  # Household refrigerators
}

scored = df_layer1.set_index("subheading")
print(f"{'Code':<8} {'Expected':>8} {'Got':>4} {'Score':>6} {'Unc':>5}  OK?  Description")
print("-" * 100)

n_correct = 0
n_found   = 0
for code, expected in ANCHOR_CODES.items():
    if code not in scored.index:
        print(f"{code:<8} {'—':>8}  not found in scored dataset")
        continue
    row  = scored.loc[code]
    got  = row["nli_class"]
    ok   = "✓" if got == expected else "✗"
    if got == expected:
        n_correct += 1
    n_found += 1
    desc = str(row["subheading_text"])[:45]
    print(f"{code:<8} {expected:>8} {got:>4} "
          f"{row['nli_continuous']:>6.3f} {row['nli_uncertainty']:>5.3f}  {ok}   {desc}")

print(f"Formal anchor accuracy: {n_correct}/{n_found} = "
      f"{n_correct/n_found:.0%}" if n_found else "No anchors found in dataset")
print("Target: ≥80% before proceeding to Layer 2.")


Code     Expected  Got  Score   Unc  OK?  Description
----------------------------------------------------------------------------------------------------
270900          1    2  3.013 0.980  ✗   Oils: petroleum oils and oils obtained from b
260111          1    1  2.823 0.976  ✓   Iron ores and concentrates: non-agglomerated
520100          1    1  2.787 0.968  ✓   Cotton: not carded or combed
270112          1    5  3.143 0.984  ✗   Coal: bituminous, whether or not pulverised, 
260600          1    2  3.094 0.967  ✗   Aluminium ores and concentrates
740811          2    2  3.150 0.952  ✓   Copper: wire, of refined copper, of which the
720917          —  not found in scored dataset
390110          2    2  3.085 0.981  ✓   Ethylene polymers: in primary forms, polyethy
520511          2    2  2.861 0.985  ✓   Cotton yarn: (not sewing thread), single, of 
760110          2    5  3.272 0.948  ✗   Aluminium: unwrought, (not alloyed)
848210          3    5  3.407 0.955  ✗   Ball bearings
85